In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(7)

# ------------------------------------------------------------
# 1. Tiny English -> German dataset
# ------------------------------------------------------------

pairs = [
    ("i like cars",      "ich mag autos"),
    ("i like cats",      "ich mag katzen"),
    ("you like cars",    "du magst autos"),
    ("you like cats",    "du magst katzen"),
    ("i see cars",       "ich sehe autos"),
    ("you see cats",     "du siehst katzen"),

    # Important examples:
    # English has 3 words, German has 4 words
    ("i am driving",     "ich fahre gerade auto"),
    ("you are driving",  "du faehrst gerade auto"),
]

BOS = "<BOS>"   # beginning of sentence
EOS = "<EOS>"   # end of sentence

# ------------------------------------------------------------
# 2. Build separate English and German vocabularies
# ------------------------------------------------------------

english_words = sorted({
    word
    for english_sentence, _ in pairs
    for word in english_sentence.split()
})

german_words = [BOS, EOS] + sorted({
    word
    for _, german_sentence in pairs
    for word in german_sentence.split()
})

eng_stoi = {word: i for i, word in enumerate(english_words)}
eng_itos = {i: word for word, i in eng_stoi.items()}

de_stoi = {word: i for i, word in enumerate(german_words)}
de_itos = {i: word for word, i in de_stoi.items()}

print("English vocabulary:")
print(eng_stoi)

print("\nGerman vocabulary:")
print(de_stoi)


def encode_english_sentence(sentence):
    return torch.tensor(
        [eng_stoi[word] for word in sentence.split()],
        dtype=torch.long
    )


def encode_german_target(sentence):
    """
    German target includes EOS.

    Example:
        "ich mag autos"

    becomes:
        ["ich", "mag", "autos", "<EOS>"]

    The EOS token teaches the decoder when to stop.
    """
    return torch.tensor(
        [de_stoi[word] for word in sentence.split()] + [de_stoi[EOS]],
        dtype=torch.long
    )


# ------------------------------------------------------------
# 3. Manual parameter creation
# ------------------------------------------------------------

def make_param(*shape, scale=0.1):
    parameter = torch.randn(*shape) * scale

    # detach().requires_grad_() makes sure it is a leaf tensor
    parameter = parameter.detach().requires_grad_()

    return parameter


embedding_dim = 16
hidden_size = 32

english_vocab_size = len(eng_stoi)
german_vocab_size = len(de_stoi)

# ------------------------------------------------------------
# 4. Two different embedding tables
# ------------------------------------------------------------

# English embedding: used by the encoder
E_eng = make_param(english_vocab_size, embedding_dim)

# German embedding: used by the decoder
E_de = make_param(german_vocab_size, embedding_dim)

# ------------------------------------------------------------
# 5. Encoder RNN parameters
# ------------------------------------------------------------

W_enc_xh = make_param(embedding_dim, hidden_size)
W_enc_hh = make_param(hidden_size, hidden_size)
b_enc_h = torch.zeros(hidden_size, requires_grad=True)

# ------------------------------------------------------------
# 6. Decoder RNN parameters
# ------------------------------------------------------------

W_dec_xh = make_param(embedding_dim, hidden_size)
W_dec_hh = make_param(hidden_size, hidden_size)
b_dec_h = torch.zeros(hidden_size, requires_grad=True)

# Decoder output layer:
# hidden state -> German vocabulary logits
W_out = make_param(hidden_size, german_vocab_size)
b_out = torch.zeros(german_vocab_size, requires_grad=True)

parameters = [
    E_eng,
    E_de,

    W_enc_xh,
    W_enc_hh,
    b_enc_h,

    W_dec_xh,
    W_dec_hh,
    b_dec_h,

    W_out,
    b_out,
]

English vocabulary:
{'am': 0, 'are': 1, 'cars': 2, 'cats': 3, 'driving': 4, 'i': 5, 'like': 6, 'see': 7, 'you': 8}

German vocabulary:
{'<BOS>': 0, '<EOS>': 1, 'auto': 2, 'autos': 3, 'du': 4, 'faehrst': 5, 'fahre': 6, 'gerade': 7, 'ich': 8, 'katzen': 9, 'mag': 10, 'magst': 11, 'sehe': 12, 'siehst': 13}


In [2]:
#The encoder reads the English sentence word by word.
def encoder(src_ids):
    """
    src_ids shape:
        [source_sentence_length]

    Example:
        "i am driving"

    src_ids length:
        3

    Returns:
        final hidden state h
    """

    h = torch.zeros(hidden_size)

    hidden_states = []

    for token_id in src_ids:

        # English word embedding
        #
        # token_id is an integer
        # E_eng[token_id] is one vector of shape [embedding_dim]
        x_t = E_eng[token_id]

        # Manual RNN equation:
        #
        # h_t = tanh(x_t W_xh + h_{t-1} W_hh + b)
        h = torch.tanh(
            x_t @ W_enc_xh +
            h   @ W_enc_hh +
            b_enc_h
        )

        hidden_states.append(h)

    hidden_states = torch.stack(hidden_states)

    # The final h is the context vector
    return h, hidden_states

In [3]:
#The decoder produces German words.
#During training, the decoder is fed the correct previous German word. This is called teacher forcing.
def decoder_train(context, tgt_ids):
    """
    context:
        final encoder hidden state

    tgt_ids:
        German target token ids, including EOS.

    Example:
        "ich fahre gerade auto <EOS>"

    Returns:
        logits over German vocabulary at each decoder step
    """

    h = context

    # First decoder input is always BOS
    prev_token_id = torch.tensor(de_stoi[BOS], dtype=torch.long)

    logits_over_time = []

    for target_token_id in tgt_ids:

        # German embedding lookup
        x_t = E_de[prev_token_id]

        # Manual decoder RNN equation
        h = torch.tanh(
            x_t @ W_dec_xh +
            h   @ W_dec_hh +
            b_dec_h
        )

        # Fully connected output layer:
        #
        # hidden state -> German word scores
        logits_t = h @ W_out + b_out

        logits_over_time.append(logits_t)

        # Teacher forcing:
        # Feed the correct previous word into the next decoder step.
        prev_token_id = target_token_id

    return torch.stack(logits_over_time)

In [4]:
#Training loop
learning_rate = 0.1
epochs = 3000

for epoch in range(epochs):

    total_loss = 0.0

    for english_sentence, german_sentence in pairs:

        src_ids = encode_english_sentence(english_sentence)
        tgt_ids = encode_german_target(german_sentence)

        # Encoder
        context, encoder_hidden_states = encoder(src_ids)

        # Decoder
        logits = decoder_train(context, tgt_ids)

        # logits shape:
        #   [target_length, german_vocab_size]
        #
        # tgt_ids shape:
        #   [target_length]
        loss = F.cross_entropy(logits, tgt_ids)

        # Clear old gradients
        for p in parameters:
            p.grad = None

        # Backpropagation through encoder and decoder time steps
        loss.backward()

        # Gradient descent
        with torch.no_grad():
            for p in parameters:
                p -= learning_rate * p.grad

        total_loss += loss.item()

    if epoch % 300 == 0:
        average_loss = total_loss / len(pairs)
        print(f"epoch {epoch:4d} | average loss {average_loss:.4f}")

epoch    0 | average loss 2.6297
epoch  300 | average loss 0.0072
epoch  600 | average loss 0.0023
epoch  900 | average loss 0.0014
epoch 1200 | average loss 0.0010
epoch 1500 | average loss 0.0007
epoch 1800 | average loss 0.0006
epoch 2100 | average loss 0.0005
epoch 2400 | average loss 0.0004
epoch 2700 | average loss 0.0004


In [5]:
#Inference / translation
#During inference, we do not know the correct previous German word.
#So the decoder feeds its own previous prediction back into itself:

# <BOS> ─► predicts "ich"
# "ich" ─► predicts "fahre"
# "fahre" ─► predicts "gerade"
# "gerade" ─► predicts "auto"
# "auto" ─► predicts "<EOS>" and stops

def translate(english_sentence, max_steps=10):

    with torch.no_grad():

        src_ids = encode_english_sentence(english_sentence)

        # Encode English sentence
        context, _ = encoder(src_ids)

        # Start decoder from encoder context
        h = context

        prev_token_id = torch.tensor(de_stoi[BOS], dtype=torch.long)

        output_words = []

        for step in range(max_steps):

            # German embedding of previous German word
            x_t = E_de[prev_token_id]

            h = torch.tanh(
                x_t @ W_dec_xh +
                h   @ W_dec_hh +
                b_dec_h
            )

            logits_t = h @ W_out + b_out

            next_token_id = int(torch.argmax(logits_t))

            if next_token_id == de_stoi[EOS]:
                break

            output_words.append(de_itos[next_token_id])

            prev_token_id = torch.tensor(next_token_id, dtype=torch.long)

        return " ".join(output_words)

In [7]:
#Test
#Because this is a tiny toy dataset, the model is mostly memorizing. That is fine here because the goal is to understand the architecture.
print("\nTranslations:")

for english_sentence, german_sentence in pairs:
    prediction = translate(english_sentence)

    print(
        f"{english_sentence:18s} -> "
        f"{prediction:28s} | target: {german_sentence}"
    )


Translations:
i like cars        -> ich mag autos                | target: ich mag autos
i like cats        -> ich mag katzen               | target: ich mag katzen
you like cars      -> du magst autos               | target: du magst autos
you like cats      -> du magst katzen              | target: du magst katzen
i see cars         -> ich sehe autos               | target: ich sehe autos
you see cats       -> du siehst katzen             | target: du siehst katzen
i am driving       -> ich fahre gerade auto        | target: ich fahre gerade auto
you are driving    -> du faehrst gerade auto       | target: du faehrst gerade auto
